# Agentic Loop 测试

模拟 Harbor 的多轮工具调用循环，用 subprocess 替代 `environment.exec()`

In [ ]:
import json
import os
import subprocess
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

MODEL = "mimo-v2.5-pro"
MAX_TURNS = 10
MAX_OUTPUT_CHARS = 4000

client = OpenAI(
    api_key=os.environ["MIMO_API_KEY"],
    base_url="https://api.xiaomimimo.com/v1",
)

SYSTEM = """You are a terminal agent operating in a Linux container.
Solve the task by executing bash commands, one at a time.
Observe each result before deciding the next command.
When the task is complete, reply with plain text instead of calling a tool."""

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "run_command",
            "description": "Execute a bash command and return its output.",
            "parameters": {
                "type": "object",
                "properties": {"command": {"type": "string"}},
                "required": ["command"],
            },
        },
    }
]

def exec_command(command: str) -> str:
    """本地执行命令，模拟 environment.exec()"""
    result = subprocess.run(
        command, shell=True, capture_output=True, text=True, timeout=30
    )
    output = result.stdout
    if result.stderr:
        output += f"\n[stderr]\n{result.stderr}"
    if result.returncode != 0:
        output += f"\n[exit code: {result.returncode}]"
    return output[:MAX_OUTPUT_CHARS] or "(no output)"

print(f"✓ 模型: {MODEL}")
print(f"✓ MAX_TURNS: {MAX_TURNS}")

## Agent Loop 函数

In [ ]:
def run_agent(task: str) -> None:
    print(f"\n{'█'*70}")
    print(f"任务: {task}")
    print(f"{'█'*70}")

    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Task:\n{task}"},
    ]

    for turn in range(1, MAX_TURNS + 1):
        print(f"\n{'─'*60}")
        print(f"Turn {turn}")
        print(f"{'─'*60}")

        # ── 发送请求体 ─────────────────────────────────────────
        request_payload = {
            "model": MODEL,
            "messages": messages,
            "tools": TOOLS,
        }
        print("\n▶ 发送请求体:")
        # messages 可能很长，只打印最后一条
        display_payload = dict(request_payload)
        display_payload["messages"] = [
            f"... ({len(messages)-1} previous messages) ...",
            messages[-1],
        ] if len(messages) > 2 else messages
        print(json.dumps(display_payload, indent=2, ensure_ascii=False, default=str))

        try:
            response = client.chat.completions.create(**request_payload)
        except Exception as e:
            print(f"\n✗ API 请求失败: {type(e).__name__}: {e}")
            break

        # ── 完整响应体 ─────────────────────────────────────────
        print("\n◀ 完整响应体:")
        print(json.dumps(response.model_dump(), indent=2, ensure_ascii=False))

        msg = response.choices[0].message
        messages.append(msg)

        # ── 判断分支 ───────────────────────────────────────────
        if not msg.tool_calls:
            print(f"\n✓ 任务完成（Turn {turn}）")
            print(f"最终回答: {msg.content}")
            break

        # ── 执行工具 ───────────────────────────────────────────
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            cmd = args["command"]

            print(f"\n⚙ 执行命令: {cmd}")
            output = exec_command(cmd)
            print(f"输出:\n{output}")

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": output,
            })
    else:
        print(f"\n⚠ 达到最大轮数 {MAX_TURNS}，任务未完成")

## 测试任务

In [ ]:
# 任务1：单步（预期 1 次工具调用）
run_agent("显示当前系统时间和主机名")

In [ ]:
# 任务2：多步（需要多次工具调用）
run_agent("找出当前目录下最大的文件，并显示其前10行内容")

In [ ]:
# 任务3：需要判断的任务（Harbor benchmark 典型场景）
run_agent("创建一个名为 test_harbor.txt 的文件，写入当前日期，然后验证文件内容正确")